# Part 6 — molecule-head prediction: the `balanced_bce` family

Same six objectives as `part_5` (`grid_0000`/`bce`, `grid_0001`/`+contractive`,
`grid_0002`/`+peak_random`, `grid_0003`/`+peak_label_invariant`,
`grid_0004`/`+contractive+peak_label_invariant`,
`grid_0005`/`grid_0004+negw_multilabel_jaccard`) — all six use the identical
`balanced_bce` head loss ([`methodology.md`](../report/methodology.md) §5), so any
difference in classification quality below is attributable to the encoder
representation each regularizer/contrastive term shapes, not to a different
head-loss function (unlike `grid_0006`'s `nnpu`, which is a genuinely different loss
and not comparable this way — out of scope here). `part_4`/`part_7` already
established that `+contractive` genuinely flattens the encoder locally and collapses
the latent cloud — this notebook is where that geometric change gets checked against
actual classification quality, together with the two new chain questions:
does `+contractive` change anything once layered onto `+peak_label_invariant`
(`grid_0003` → `grid_0004`), and does `+negw_multilabel_jaccard` change anything on
top of that (`grid_0004` → `grid_0005`)?

Reuses `part_2`'s fixed `evaluate_head`/`per_class_metrics` (the per-class
`(sample, class)` mask handling — `analysis.autoencoder.heads.metrics`) and its
all-zero-baseline framing for why `average_precision` (macro AUPRC), not
`hamming_loss`/accuracy-shaped metrics, is the number to trust here.

**Scope**: all 5 repetitions on the **test** split (the primary, held-out read,
matching `part_2`'s rigor); repetition 0 additionally evaluated on **train** as well,
to read the generalization gap directly rather than relying on total-loss curves alone
(validation is not needed for this comparison, per this campaign's scope notes). This
notebook's closing notes are, together with `part_5`'s, the intended "final conclusion
for the balanced_bce family."


In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

current_path = Path.cwd().resolve()
repository_root = next(path for path in (current_path, *current_path.parents) if (path / "pyproject.toml").is_file())
os.chdir(repository_root)

from msi_autoencoder_wrapper import MSIAutoEncoderWrapper
from msi_autoencoder_wrapper.analysis.autoencoder.heads.metrics import evaluate_head, per_class_metrics, probabilities_from_logits
from msi_autoencoder_wrapper.visualization import resolve_theme


2026-08-26 11:50:54,764 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.binners_strategies'.


2026-08-26 11:50:54,768 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 6 implementation module(s) in package 'msi_autoencoder_wrapper.binners.inverse_strategies'.


2026-08-26 11:50:55,492 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 3 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders.presets'.


2026-08-26 11:50:55,494 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 22 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.autoencoders'.


2026-08-26 11:50:55,495 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 0 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures.types.schema'.


2026-08-26 11:50:55,496 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 29 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures'.


2026-08-26 11:50:55,511 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 2 implementation module(s) in package 'msi_autoencoder_wrapper.models.datasets.strategies'.


2026-08-26 11:50:55,525 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 14 implementation module(s) in package 'msi_autoencoder_wrapper.training.criterions.autoencoder'.


2026-08-26 11:50:55,526 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 18 implementation module(s) in package 'msi_autoencoder_wrapper.training.criterions'.


## Config

In [2]:
WORKSPACE = Path("data/kidney_workspace")
MODEL_CONTEXT = "kidney"
CAMPAIGN_ID = "kidney-architecture-predictive__cfg_529dc0e4494b"
GRID_ID_BY_OBJECTIVE = {
    "bce": "grid_0000",
    "contractive": "grid_0001",
    "peak_random": "grid_0002",
    "peak_label_invariant": "grid_0003",
    "grid4": "grid_0004",
    "grid5": "grid_0005",
}
OBJECTIVE_LABELS = {
    "bce": "bce",
    "contractive": "+contractive",
    "peak_random": "+peak_random",
    "peak_label_invariant": "+peak_label_invariant",
    "grid4": "+contractive+peak_label_invariant",
    "grid5": "+contractive+peak_label_invariant+jaccard",
}

# Curated comparison pairs (not all C(6,2)=15): each single-term objective vs. bce,
# plus the two chain links this notebook adds — matches part_5's PAIRS.
PAIRS = [
    ("bce", "contractive"),
    ("bce", "peak_random"),
    ("bce", "peak_label_invariant"),
    ("peak_label_invariant", "grid4"),
    ("grid4", "grid5"),
    ("bce", "grid4"),
    ("bce", "grid5"),
]

REPETITIONS = range(5)
HEAD_NAME = "molecule_primary"
TARGET_FIELD = "molecule"
THRESHOLD = 0.5
BATCH_SIZE = 256

theme = resolve_theme(None)
theme.apply()


## Environment workaround (same as `part_2`)


In [3]:
SCRATCH_WORKSPACE_ROOT = Path("/tmp/ms488923/msi-wrapper/predictive-20260824-04/kidney_workspace")
SCRATCH_WORKSPACE_ROOT.parent.mkdir(parents=True, exist_ok=True)
if not SCRATCH_WORKSPACE_ROOT.is_symlink():
    SCRATCH_WORKSPACE_ROOT.symlink_to(WORKSPACE.resolve())


## Evaluate every (objective, repetition) on test; repetition 0 also on train


In [4]:
wrapper = MSIAutoEncoderWrapper(project_path=str(WORKSPACE))
wrapper.workspace.set_default_image_path(MODEL_CONTEXT)


def evaluate_split(model, partitions, split_name: str) -> dict:
    device = next(model.parameters()).device
    dataset = getattr(partitions, split_name)
    logits_batches, target_batches, mask_batches = [], [], []
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    with torch.no_grad():
        for _, spectra, targets, masks in loader:
            outputs = model(spectra.float().to(device))
            logits_batches.append(outputs[f"head_{HEAD_NAME}"].cpu().numpy())
            target_batches.append(targets[TARGET_FIELD].numpy())
            mask_batches.append(masks[TARGET_FIELD].numpy())
    return {
        "logits": np.concatenate(logits_batches, axis=0),
        "targets": np.concatenate(target_batches, axis=0),
        "mask": np.concatenate(mask_batches, axis=0),
        "size": len(dataset),
    }


# All six objectives share one image/binner/annotation-store/dataset context
# (verified in part_5) — only the model weights differ between
# grid_0000/0001/0002/0003/0004/0005 and their repetitions. `load_configuration`
# unconditionally re-resolves the raw imzML reader and annotation index on every
# call (measured ~37-40s), even when nothing about the image actually changed —
# a real cost sink at 30 model loads.
# `models_manager.load_model` swaps only the model architecture/weights (its own
# docstring: "does not require loading its original image") and leaves
# `active_dataset`/context untouched — measured ~0.04s. So: pay the full context
# cost exactly once, then use the lightweight swap for every other model.
first_objective, first_grid_id = next(iter(GRID_ID_BY_OBJECTIVE.items()))
wrapper.load_configuration(model_name=f"{CAMPAIGN_ID}__{first_grid_id}__rep_00")
shared_partitions = wrapper.active_dataset.create_partitions()

results = {}
for objective, grid_id in GRID_ID_BY_OBJECTIVE.items():
    for repetition in REPETITIONS:
        model_name = f"{CAMPAIGN_ID}__{grid_id}__rep_0{repetition}"
        if (objective, repetition) == (first_objective, 0):
            model = wrapper.active_model  # already loaded above
        else:
            model = wrapper.models_manager.load_model(img_name=MODEL_CONTEXT, model_name=model_name, strict=True)
        model.eval()
        results[(objective, repetition, "test")] = evaluate_split(model, shared_partitions, "test")
        if repetition == 0:
            results[(objective, repetition, "train")] = evaluate_split(model, shared_partitions, "train")
        print(f"{objective} rep {repetition}: done")


2026-08-26 11:50:56,255 | INFO     | msi_autoencoder_wrapper.core.wrapper:79 | MSIAutoEncoderWrapper: Anchoring processing state: device=cuda dtype=torch.float32


2026-08-26 11:50:56,256 | INFO     | msi_autoencoder_wrapper.core.mixins.context_manager.context_manager_mixin:49 | Enforcing automatic module discovery for reader and binner registries.


2026-08-26 11:50:56,257 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 2 implementation module(s) in package 'msi_autoencoder_wrapper.readers.strategies'.


2026-08-26 11:50:56,258 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 1 implementation module(s) in package 'msi_autoencoder_wrapper.binners.binners_strategies'.


2026-08-26 11:50:56,259 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 6 implementation module(s) in package 'msi_autoencoder_wrapper.binners.inverse_strategies'.


...

2026-08-26 11:57:33,403 | INFO     | msi_autoencoder_wrapper.models.architectures.architectures_manager:235 | Injecting resolved computational components ledger dictionary into the master architecture graph.


2026-08-26 11:57:33,404 | INFO     | msi_autoencoder_wrapper.models.architectures.types.autoencoders.base_architecture_autoencoder:49 | MSIBaseAutoencoderArchitecture master network graph successfully assembled.


2026-08-26 11:57:33,407 | INFO     | msi_autoencoder_wrapper.utils.module_search:79 | Discovered 29 implementation module(s) in package 'msi_autoencoder_wrapper.models.architectures'.


2026-08-26 11:57:33,414 | INFO     | msi_autoencoder_wrapper.core.mixins.models_manager.proxies.model_runtime_proxy:108 | Attached loaded model 'conv1d-ae-32-16-8-latent-10' as family 'autoencoder'.


grid5 rep 4: done


## Aggregate metrics — mean/std across repetitions, test split


In [5]:
aggregate_rows = []
for objective in GRID_ID_BY_OBJECTIVE:
    for repetition in REPETITIONS:
        result = results[(objective, repetition, "test")]
        metrics = evaluate_head(result["logits"], result["targets"], "multi_label", result["mask"], THRESHOLD)
        aggregate_rows.append({"objective": OBJECTIVE_LABELS[objective], "repetition": repetition, "split": "test", **metrics})

aggregate_frame = pd.DataFrame(aggregate_rows)
aggregate_frame.groupby("objective")[["micro_f1", "macro_f1", "hamming_loss", "average_precision"]].agg(["mean", "std"])


micro_f1            macro_f1  \
                                               mean       std      mean   
objective                                                                 
+contractive                               0.121633  0.011748  0.043745   
+contractive+peak_label_invariant          0.125165  0.012451  0.044669   
+contractive+peak_label_invariant+jaccard  0.124330  0.012505  0.043659   
+peak_label_invariant                      0.115111  0.012344  0.046680   
+peak_random                               0.122736  0.010897  0.049268   
bce                                        0.118053  0.008894  0.047808   

                                                    hamming_loss            \
                                                std         mean       std   
objective                                                                    
+contractive                               0.005391     0.316720  0.014917   
+contractive+peak_label_invariant          0.004812     0.313666  0.011936   
+contractive+peak_label_invariant+jaccard  0.005068     0.313336  0.012660   
+peak_label_invariant                      0.005747     0.330399  0.018001   
+peak_random                               0.004482     0.327895  0.015205   
bce                                        0.004130     0.328477  0.011244   

                                          average_precision            
                                                       mean       std  
objective                                                              
+contractive                                       0.353019  0.009034  
+contractive+peak_label_invariant                  0.357359  0.010295  
+contractive+peak_label_invariant+jaccard          0.352058  0.016371  
+peak_label_invariant                              0.363741  0.019750  
+peak_random                                       0.358037  0.024198  
bce                                                0.364260  0.016666

In [6]:
aggregate_frame.sort_values(["objective", "repetition"]).reset_index(drop=True)


,objective,repetition,split,micro_f1,macro_f1,micro_precision,micro_recall,hamming_loss,average_precision
0,+contractive,0,test,0.126515,0.046706,0.075051,0.402570,0.319040,0.345249
1,+contractive,1,test,0.112651,0.039501,0.066581,0.365671,0.330630,0.356891
2,+contractive,2,test,0.133058,0.044818,0.080024,0.394513,0.295055,0.366928
3,+contractive,3,test,0.105925,0.037207,0.063499,0.319198,0.309261,0.346076
4,+contractive,4,test,0.130017,0.050494,0.076615,0.429144,0.329613,0.349949
5,+contractive+peak_label_invariant,0,test,0.137141,0.048052,0.081656,0.427874,0.309016,0.363243
6,+contractive+peak_label_invariant,1,test,0.115607,0.040468,0.068539,0.369030,0.324054,0.354227
7,+contractive+peak_label_invariant,2,test,0.132720,0.046602,0.079702,0.396414,0.297348,0.362680
8,+contractive+peak_label_invariant,3,test,0.108377,0.038672,0.064850,0.329608,0.311268,0.365946
9,+contractive+peak_label_invariant,4,test,0.131981,0.049550,0.077866,0.432673,0.326641,0.340702


## Is the between-objective difference bigger than seed noise?

Same check as `part_5` Stage 1b and `part_7`'s corrected geometry comparison, applied
here from the start rather than retrofitted: within-objective `|rep_i - rep_j|` on
`average_precision` (the metric `part_2` established as the one to trust for this
head) as the null distribution, checked against each non-`bce` objective's distance
from `bce`.


In [7]:
from itertools import combinations

primary_metric = "average_precision"

null_rows = []
for objective in GRID_ID_BY_OBJECTIVE:
    values = aggregate_frame.loc[aggregate_frame["objective"] == OBJECTIVE_LABELS[objective], primary_metric].to_numpy()
    reps = aggregate_frame.loc[aggregate_frame["objective"] == OBJECTIVE_LABELS[objective], "repetition"].to_numpy()
    for (i, value_a), (j, value_b) in combinations(zip(reps, values), 2):
        null_rows.append({"objective": OBJECTIVE_LABELS[objective], "rep_pair": f"{i}-{j}", "abs_diff": abs(value_a - value_b)})

null_frame = pd.DataFrame(null_rows)
null_summary = null_frame.groupby("objective")["abs_diff"].agg(["mean", "std", "min", "max"])

objective_means = aggregate_frame.groupby("objective")[primary_metric].mean()
print(f"within-objective |rep_i - rep_j| null ({primary_metric}):")
print(null_summary)
print()

comparison_rows = []
for left_key, right_key in PAIRS:
    left_label, right_label = OBJECTIVE_LABELS[left_key], OBJECTIVE_LABELS[right_key]
    pair_label = f"{left_label} vs {right_label}"
    between = abs(objective_means[left_label] - objective_means[right_label])
    larger_null = max(null_summary.loc[left_label, "mean"], null_summary.loc[right_label, "mean"])
    comparison_rows.append({
        "pair": pair_label,
        "between_mean": between,
        "larger_null_mean": larger_null,
        "signal_over_larger_null": between / larger_null,
    })
    print(f"between-objective |mean({left_label}) - mean({right_label})|: {between:.4f} (signal/null = {between / larger_null:.2f})")

comparison_frame = pd.DataFrame(comparison_rows)
comparison_frame


within-objective |rep_i - rep_j| null (average_precision):
                                               mean       std       min  \
objective                                                                 
+contractive                               0.010835  0.007137  0.000827   
+contractive+peak_label_invariant          0.011901  0.008841  0.000563   
+contractive+peak_label_invariant+jaccard  0.020059  0.012185  0.002973   
+peak_label_invariant                      0.024145  0.014800  0.003885   
+peak_random                               0.029595  0.018113  0.003700   
bce                                        0.020436  0.012379  0.001260   

                                                max  
objective                                            
+contractive                               0.021679  
+contractive+peak_label_invariant          0.025244  
+contractive+peak_label_invariant+jaccard  0.036050  
+peak_label_invariant                      0.054515  
+peak_random    

,pair,between_mean,larger_null_mean,signal_over_larger_null
0,bce vs +contractive,0.011241,0.020436,0.550050
1,bce vs +peak_random,0.006222,0.029595,0.210257
2,bce vs +peak_label_invariant,0.000519,0.024145,0.021484
3,+peak_label_invariant vs +contractive+peak_lab...,0.006381,0.024145,0.264292
4,+contractive+peak_label_invariant vs +contract...,0.005301,0.020059,0.264278
5,bce vs +contractive+peak_label_invariant,0.006900,0.020436,0.337643
6,bce vs +contractive+peak_label_invariant+jaccard,0.012201,0.020436,0.597050


## Train vs. test — repetition 0

Same metrics computed on the pixels each model actually trained on, for a direct
generalization-gap read per objective (analogous to `part_1`'s train/validation
Masserstein gap, but for the classification head).


In [8]:
gap_rows = []
for objective in GRID_ID_BY_OBJECTIVE:
    for split_name in ("train", "test"):
        result = results[(objective, 0, split_name)]
        metrics = evaluate_head(result["logits"], result["targets"], "multi_label", result["mask"], THRESHOLD)
        gap_rows.append({"objective": OBJECTIVE_LABELS[objective], "split": split_name, **metrics})

gap_frame = pd.DataFrame(gap_rows)
gap_frame[["objective", "split", "macro_f1", "average_precision", "hamming_loss"]]


,objective,split,macro_f1,average_precision,hamming_loss
0,bce,train,0.232623,0.613690,0.327261
1,bce,test,0.051750,0.385860,0.323778
2,+contractive,train,0.216587,0.525394,0.323671
3,+contractive,test,0.046706,0.345249,0.319040
4,+peak_random,train,0.231223,0.616147,0.327670
5,+peak_random,test,0.055527,0.384673,0.320970
6,+peak_label_invariant,train,0.233808,0.626786,0.323412
7,+peak_label_invariant,test,0.055857,0.391997,0.322469
8,+contractive+peak_label_invariant,train,0.216022,0.526347,0.309263
9,+contractive+peak_label_invariant,test,0.048052,0.363243,0.309016


## Per-class detail, test split, repetition 0

Same convention as `part_2`: classes with zero test positives are excluded (AUPRC is
undefined there, not zero).


In [9]:
class_metrics_by_objective = {}
for objective in GRID_ID_BY_OBJECTIVE:
    result = results[(objective, 0, "test")]
    probabilities = probabilities_from_logits(result["logits"], "multi_label")
    frame = pd.DataFrame(per_class_metrics(probabilities, result["targets"], THRESHOLD, result["mask"]))
    class_metrics_by_objective[objective] = frame[frame["positive_samples"] > 0].copy()
    print(f"{objective}: {len(class_metrics_by_objective[objective])} classes scored")


bce: 113 classes scored


contractive: 113 classes scored


peak_random: 113 classes scored


peak_label_invariant: 113 classes scored


grid4: 113 classes scored


grid5: 113 classes scored


In [10]:
comparison = pd.DataFrame({
    objective: class_metrics_by_objective[objective].set_index("class_index")["average_precision"]
    for objective in GRID_ID_BY_OBJECTIVE
}).dropna(how="all")
for left_key, right_key in PAIRS:
    comparison[f"{left_key}_minus_{right_key}"] = comparison[left_key] - comparison[right_key]
comparison.describe()


,bce,contractive,peak_random,peak_label_invariant,grid4,grid5,bce_minus_contractive,bce_minus_peak_random,bce_minus_peak_label_invariant,peak_label_invariant_minus_grid4,grid4_minus_grid5,bce_minus_grid4,bce_minus_grid5
count,113.000000,113.000000,113.000000,113.000000,113.000000,113.000000,113.000000,113.000000,113.000000,113.000000,113.000000,113.000000,113.000000
mean,0.385860,0.345249,0.384673,0.391997,0.363243,0.347700,0.040610,0.001187,-0.006137,0.028754,0.015543,0.022617,0.038160
std,0.297384,0.255279,0.310012,0.300883,0.264936,0.253740,0.286505,0.172261,0.150770,0.288827,0.063103,0.302590,0.296855
min,0.000452,0.000402,0.000503,0.000506,0.000398,0.000415,-0.662684,-0.604577,-0.484339,-0.697690,-0.099612,-0.716942,-0.696440
25%,0.152974,0.179471,0.129175,0.146148,0.198149,0.186436,-0.072195,-0.049214,-0.025320,-0.154235,-0.003520,-0.128412,-0.122341
50%,0.323596,0.285278,0.285154,0.324141,0.291521,0.288927,0.005587,0.004689,0.000993,0.000108,0.000649,0.001956,0.005454
75%,0.608920,0.457532,0.598308,0.636234,0.492453,0.488962,0.196259,0.072742,0.043494,0.147169,0.019379,0.198195,0.218468
max,0.999912,0.981925,0.995477,0.993657,0.998438,0.996427,0.769026,0.535481,0.494058,0.763162,0.446612,0.832058,0.830147


## Notes for the next pass

**Headline: classification quality stays flat across the entire balanced_bce
family — every one of the 7 curated comparisons, including the two new chain
links, fails the seed-noise null check.** In particular, `+contractive`'s
reconstruction *regression* when added to `+peak_label_invariant` (`part_5`) does
**not** repeat here: `+peak_label_invariant` → `grid_0004` shows no detectable
classification change either direction.

### Result (all 5 repetitions, test AUPRC, null-checked from the start)

| objective | mean AUPRC | std | vs. predecessor | signal/null |
|---|---|---|---|---|
| `bce` | 0.3643 | 0.0167 | — | — |
| `+contractive` | 0.3530 | 0.0090 | vs. `bce`: 0.0112 | 0.55 (no) |
| `+peak_random` | 0.3580 | 0.0242 | vs. `bce`: 0.0062 | 0.21 (no) |
| `+peak_label_invariant` | **0.3637** | 0.0198 | vs. `bce`: 0.0005 | 0.02 (no) |
| `+contractive+peak_label_invariant` (`grid_0004`) | 0.3574 | 0.0103 | vs. `+peak_label_invariant`: 0.0064 | 0.26 (no) |
| `+contractive+peak_label_invariant+jaccard` (`grid_0005`) | 0.3521 | 0.0164 | vs. `grid_0004`: 0.0053 | 0.26 (no) |

- **Every comparison stays below the null**, `bce` vs. `+peak_label_invariant`
  most emphatically (ratio 0.02 — the two means are 0.0005 apart, essentially
  identical). The two new chain links (`+peak_label_invariant`→`grid_0004`,
  `grid_0004`→`grid_0005`) land at the same modest 0.26 ratio as each other —
  neither `+contractive` nor `+negw_multilabel_jaccard` produces a detectable
  classification effect when layered on.
- **The train-AUPRC pattern from the 4-objective pass reproduces cleanly and
  strengthens the earlier "lead."** Repetition 0 train AUPRC: `+contractive`
  alone 0.525; `grid_0004` (`+contractive` layered onto `+peak_label_invariant`)
  0.526 — essentially identical to plain `+contractive`'s train fit, *not* to
  `+peak_label_invariant`'s own train fit (0.627). Whatever `+contractive` does to
  training-set fit, it reproduces almost exactly when combined with
  `+peak_label_invariant`, independent of that term. `grid_0005`'s train AUPRC
  (0.519) continues the same level. Still single-repetition, not null-checked, so
  a lead rather than a conclusion — but a strikingly consistent one across two
  independent objectives that both include `+contractive`.
- **Per-class detail**: `peak_label_invariant_minus_grid4` mean 0.029 (small,
  `+peak_label_invariant` slightly ahead, same direction as the aggregate table)
  but std 0.289 — an order of magnitude larger, consistent with the null check
  finding no reliable effect. `grid4_minus_grid5` mean 0.016, std 0.063 — also
  noise-dominated.

**Combined with `part_5`**: reconstruction and classification diverge sharply for
`+contractive`+`+peak_label_invariant`. Adding `+contractive` to
`+peak_label_invariant` *measurably hurts reconstruction* (`part_5`, signal/null
1.12) but leaves classification statistically untouched (this notebook, 0.26).
That asymmetry, together with the matching train-AUPRC lead here, points toward
`+contractive`'s cost showing up specifically in how well the encoder retains
reconstructible detail, not in the class-discriminative signal the head actually
uses — worth checking directly against `part_7`'s dimension-usage/RSA numbers once
that notebook is re-run with `grid_0004`/`grid_0005` included.

**Provisional balanced_bce-family conclusion for classification**: no objective in
this family is distinguishable from `bce` on classification quality at 5
repetitions. Combined with `part_5`'s reconstruction verdict
(`+peak_label_invariant` alone is best; `+contractive` on top is a real
regression), the balanced_bce-family recommendation is `grid_0003`
(`+peak_label_invariant` alone) over `grid_0004`/`grid_0005` — the additional
terms in the latter two do not help classification and measurably hurt
reconstruction.
